# RxAssist AI - Contraindication Retrieval Pipeline

This notebook runs the complete contraindication similarity search pipeline using Jina v3 embeddings on Google Colab. It processes pharmaceutical contraindications and finds similar medical conditions using vector embeddings.

## What this notebook does:
1. 🔧 Sets up the environment and installs dependencies
2. 📁 Creates project structure and handles file uploads
3. 🎯 Runs the retrieval pipeline with Jina v3 embeddings
4. 🔗 Builds interaction matrices from the results
5. 📊 Provides comprehensive statistics and downloads

## Requirements:
- Vector database (created with Jina v3 embeddings)
- Contraindications data file
- Source code files

## 1. Environment Setup

Install required packages and check GPU availability.

In [ ]:
# Install only essential dependencies
!pip install -q chromadb
!pip install -q torch
!pip install -q tqdm
!pip install -q scipy
!pip install -q numpy

# Check GPU availability
import torch
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎯 Device: {torch.cuda.get_device_name(0)}")
else:
    print("💻 Using CPU")

print("\n✅ Environment setup complete!")

## 2. Mount Google Drive and Setup Project Structure

In [ ]:
# Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Create project directory structure
project_dirs = [
    "data/contraindications",
    "data/interaction_results", 
    "data/interaction_matrix",
    "data/vector_dbs/vector_db_jinaai_jina_embeddings_v3/chroma_langchain_db",
    "scripts",
    "src/retrieval"
]

for directory in project_dirs:
    os.makedirs(directory, exist_ok=True)
    print(f"📁 Created: {directory}")

print("\n✅ Project structure created!")

## 3. File Upload Options

Choose one of the following methods to upload your files:

### Option A: Upload from Local Computer
Use the file manager panel on the left to upload:
- `all_contraindications_verified.json` → `data/contraindications/`
- `vector_db_jinaai_jina_embeddings_v3/` folder → `data/vector_dbs/`
- `3_run_retrieval.py` → `scripts/`
- `vector_db_retrieval.py` → `src/retrieval/`
- `interaction_matrix.py` → `src/retrieval/`

### Option B: Copy from Google Drive
If your files are in Google Drive, modify and run the cell below:

In [ ]:
# Option B: Copy files from Google Drive
import shutil

# 🔧 MODIFY THESE PATHS to match your Google Drive structure
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/rxassist-ai"  # Change this!

# Uncomment and modify the paths below if using Google Drive:

# Copy contraindications data
# shutil.copy(f"{DRIVE_PROJECT_PATH}/data/contraindications/all_contraindications_verified.json", 
#             "data/contraindications/")

# Copy vector database
# shutil.copytree(f"{DRIVE_PROJECT_PATH}/data/vector_dbs/vector_db_jinaai_jina_embeddings_v3", 
#                 "data/vector_dbs/vector_db_jinaai_jina_embeddings_v3", dirs_exist_ok=True)

# Copy scripts and source code
# shutil.copy(f"{DRIVE_PROJECT_PATH}/scripts/3_run_retrieval.py", "scripts/")
# shutil.copy(f"{DRIVE_PROJECT_PATH}/src/retrieval/vector_db_retrieval.py", "src/retrieval/")
# shutil.copy(f"{DRIVE_PROJECT_PATH}/src/retrieval/interaction_matrix.py", "src/retrieval/")

print("📥 Files ready for copying from Google Drive!")
print("💡 Uncomment and modify the paths above to use this option")

## 4. Verify Required Files

In [ ]:
# Check essential files
import os

required_files = [
    "data/contraindications/all_contraindications_verified.json",
    "scripts/3_run_retrieval.py", 
    "src/retrieval/vector_db_retrieval.py",
    "src/retrieval/interaction_matrix.py",
    "data/vector_dbs/vector_db_jinaai_jina_embeddings_v3/chroma_langchain_db"
]

print("🔍 Checking required files...")
all_present = True

for file_path in required_files:
    if os.path.exists(file_path):
        print(f"✅ {file_path}")
    else:
        print(f"❌ {file_path}")
        all_present = False

if all_present:
    print("\n🎉 All files present! Ready to run.")
else:
    print("\n⚠️ Missing files. Upload them first.")

## 5. Configuration

Set up the retrieval parameters. Modify these values if needed:

In [ ]:
# Configuration
MODEL_NAME = "jinaai/jina-embeddings-v3"
AIC_CODES = None  # Set to ["045034307"] to filter specific codes
CATEGORY_FILTER = None  # Set to "condition" to filter by category

print(f"📊 Configuration:")
print(f"   Model: {MODEL_NAME}")
print(f"   AIC Filter: {AIC_CODES or 'All codes'}")
print(f"   Category Filter: {CATEGORY_FILTER or 'All categories'}")

# Example: AIC_CODES = ["045034307"]  # Uncomment to filter

## 6. Run Contraindication Retrieval Pipeline

This will run the complete pipeline including similarity search and interaction matrix building:

In [ ]:
# Run the pipeline
import subprocess
import sys

print("🚀 Starting pipeline...")

# Build command
cmd = [sys.executable, "scripts/3_run_retrieval.py", "--model", MODEL_NAME]
if AIC_CODES:
    cmd.extend(["--aic-codes"] + AIC_CODES)
if CATEGORY_FILTER:
    cmd.extend(["--category", CATEGORY_FILTER])

print(f"💻 Running: {' '.join(cmd)}")
print("-" * 50)

# Run with real-time output
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in process.stdout:
    print(line.rstrip())

return_code = process.wait()

if return_code == 0:
    print("✅ Pipeline completed!")
else:
    print(f"❌ Failed with code: {return_code}")

## 7. Explore Results

Let's examine the generated files and their contents:

In [ ]:
# Check generated files
import os
import glob

print("📁 Generated files:")

# Results
results = glob.glob("data/interaction_results/*.json")
if results:
    latest = max(results, key=os.path.getctime)
    size = os.path.getsize(latest) / (1024*1024)
    print(f"📊 Results: {os.path.basename(latest)} ({size:.1f} MB)")
else:
    print("❌ No results found")

# Matrix
matrix = glob.glob("data/interaction_matrix/*.json")
if matrix:
    latest = max(matrix, key=os.path.getctime)
    size = os.path.getsize(latest) / (1024*1024)
    print(f"🔗 Matrix: {os.path.basename(latest)} ({size:.1f} MB)")
else:
    print("❌ No matrix found")

## 8. Analyze Results

Load and analyze the most recent results:

In [ ]:
# Analyze latest results
import json
import glob

results_files = glob.glob("data/interaction_results/*.json")
if results_files:
    latest = max(results_files, key=os.path.getctime)
    
    with open(latest, 'r') as f:
        results = json.load(f)
    
    aic_results = results.get('aic_results', [])
    total_contraindications = sum(len(aic.get('similarity_searches', [])) for aic in aic_results)
    total_matches = sum(
        sum(len(search.get('similar_documents', [])) for search in aic.get('similarity_searches', []))
        for aic in aic_results
    )
    
    print(f"📊 Results Summary:")
    print(f"   AICs processed: {len(aic_results)}")
    print(f"   Contraindications: {total_contraindications:,}")
    print(f"   ICD matches: {total_matches:,}")
    if total_contraindications > 0:
        print(f"   Avg matches/contraindication: {total_matches/total_contraindications:.1f}")
        
else:
    print("❌ No results to analyze")

## 9. Download Results

Download the generated files for further analysis:

In [ ]:
# Download results
from google.colab import files
import os
import glob

print("⬇️ Downloading results...")

# Download latest results file
results = glob.glob("data/interaction_results/*.json")
if results:
    latest = max(results, key=os.path.getctime)
    print(f"📊 Downloading: {os.path.basename(latest)}")
    files.download(latest)

# Download latest matrix file
matrix = glob.glob("data/interaction_matrix/*.json")
if matrix:
    latest = max(matrix, key=os.path.getctime)
    print(f"🔗 Downloading: {os.path.basename(latest)}")
    files.download(latest)

print("✅ Download complete!")

# ✅ Pipeline Complete!

You've successfully run the RxAssist AI contraindication retrieval pipeline with Jina v3 embeddings.

## What was done:
1. ✅ Processed pharmaceutical contraindications  
2. ✅ Found similar ICD-11 medical conditions
3. ✅ Built interaction matrices
4. ✅ Downloaded results for analysis

## Generated files:
- **Interaction Results**: Detailed similarity search results
- **Interaction Matrix**: Structured AIC ↔ ICD-11 mappings

## Next steps:
- Analyze the downloaded JSON files
- Use interaction matrices for healthcare applications
- Experiment with different AIC codes or categories

**Pipeline powered by Jina v3 embeddings** 🚀